# NeuralDream: EEG to Image DecodingMapping 17-channel EEG signals to **768-d CLIP ViT-Large** space for zero-shot image retrieval & conditioned synthesis.**Dataset:** THINGS-EEG2 (Single Subject: sub-01)- Train: 1,500 concepts × 10 images = 15,000 samples- Validation: 154 concepts × 10 images = 1,540 samples (used for Grid Search & Checkpointing)- Test: 200 hold-out concepts × 1 image = 200 samples (strictly unseen until final benchmark)**Pipeline & Architectural Defenses:**1. **Zero-Leakage Validation Split:** Carve out 10% of training concepts for hyperparameter tuning & early checkpointing.2. **Architecture-Aware Grid Search:** Independent block-size searches for RNNs (GRU) vs. Attention (Transformer).3. **Collision-Masked Dual Loss:** Cosine Alignment + InfoNCE with false-negative concept masking (τ=0.07).4. **Data Augmentations:** Gaussian noise (σ=0.05) + temporal masking (15% slice).5. **Image Synthesis Integration:** 768-d latents prepared for unCLIP / IP-Adapter / Image Variations pipelines.

In [ ]:
# Install dependencies (uncomment if needed)
!pip install torch torchvision transformers pillow scikit-learn matplotlib tqdm

In [ ]:
import os
import math
import warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel

warnings.filterwarnings('ignore')

In [ ]:
# Device: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')

# Reproducibility Seed
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Directories
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
RAW_EEG_DIR = os.path.join(BASE_DIR, 'data', 'raw_eeg')
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'data', 'images', 'train')
TEST_IMG_DIR = os.path.join(BASE_DIR, 'data', 'images', 'test')
CACHE_DIR = os.path.join(BASE_DIR, 'cache')
CKPT_DIR = os.path.join(BASE_DIR, 'checkpoints')
VEC_DIR = os.path.join(BASE_DIR, 'vectors')

for d in [CACHE_DIR, CKPT_DIR, VEC_DIR]:
    os.makedirs(d, exist_ok=True)

SUBJECT = 'sub-01'
print(f'Base Dir: {BASE_DIR}')
print(f'Subject: {SUBJECT}')

In [ ]:
# Data dimensions
N_CHANNELS = 17
N_TIMEPOINTS = 100
CLIP_DIM = 768  # ViT-Large/14 output dimension (native to Stable Diffusion unCLIP/IP-Adapter)

# Training
BATCH_SIZE = 128
EPOCHS = 100
LR = 5e-4
DROPOUT = 0.3
WARMUP_EPOCHS = 10  # Linear warmup before cosine decay

# Augmentations (training only)
NOISE_SIGMA = 0.05   # Gaussian noise std (5% of z-scored signal)
MASK_RATIO = 0.15    # Fraction of timepoints to zero-out per sample

# InfoNCE
TEMPERATURE = 0.07   # Controls sharpness of contrastive softmax

# Grid search: biologically grounded windows (50ms, 100ms, 200ms, 250ms)
SEARCH_EPOCHS = 20
BLOCK_SIZES = [5, 10, 20, 25]

In [ ]:
def safe_load(path):
    """Load .npy handling THINGS-EEG2 dict format."""
    arr = np.load(path, allow_pickle=True)
    if arr.dtype == object:
        arr = arr.item()
    if isinstance(arr, dict):
        if 'preprocessed_eeg_data' in arr:
            arr = arr['preprocessed_eeg_data']
        elif 'data' in arr:
            arr = arr['data']
        else:
            arr = list(arr.values())[0]
    return np.array(arr)

# Load single subject
sub_dir = os.path.join(RAW_EEG_DIR, SUBJECT)
train_file = os.path.join(sub_dir, 'preprocessed_eeg_training.npy')
test_file = os.path.join(sub_dir, 'preprocessed_eeg_test.npy')

assert os.path.exists(train_file), f'Not found: {train_file}'
assert os.path.exists(test_file), f'Not found: {test_file}'

# Average across repetitions to isolate phase-locked Event-Related Potentials (ERPs)
train_eeg = safe_load(train_file).mean(axis=1)  # (16540, 17, 100)
test_eeg = safe_load(test_file).mean(axis=1)     # (200, 17, 100)

# Baseline correction: subtract pre-stimulus mean (first 20 points = -200ms to 0ms)
BASELINE_PTS = 20
train_eeg = train_eeg - train_eeg[:, :, :BASELINE_PTS].mean(axis=-1, keepdims=True)
test_eeg = test_eeg - test_eeg[:, :, :BASELINE_PTS].mean(axis=-1, keepdims=True)

print(f'{SUBJECT} | Train: {train_eeg.shape} | Test: {test_eeg.shape}')

In [ ]:
# Z-score normalize per channel strictly on the 15,000 training subset (zero leakage)
VAL_CONCEPTS = 154
VAL_SAMPLES = VAL_CONCEPTS * 10  # 1,540 samples

mean = train_eeg[:-VAL_SAMPLES].mean(axis=(0, 2), keepdims=True)
std = train_eeg[:-VAL_SAMPLES].std(axis=(0, 2), keepdims=True) + 1e-8

train_eeg = (train_eeg - mean) / std
test_eeg = (test_eeg - mean) / std

# Transpose to (batch, timepoints, channels) for sequence models
train_eeg = train_eeg.transpose(0, 2, 1)  # (16540, 100, 17)
test_eeg = test_eeg.transpose(0, 2, 1)    # (200, 100, 17)

print(f'Normalized | Train Pool: {train_eeg.shape} | Test: {test_eeg.shape}')

In [ ]:
def get_concept_image_paths(img_dir):
    concepts = sorted([c for c in os.listdir(img_dir) if not c.startswith('.')])
    images = []
    for concept in concepts:
        folder = os.path.join(img_dir, concept)
        imgs = sorted([os.path.join(folder, f) for f in os.listdir(folder)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        images.append(imgs)
    return concepts, images

train_concepts, train_concept_imgs = get_concept_image_paths(TRAIN_IMG_DIR)
test_concepts, test_concept_imgs = get_concept_image_paths(TEST_IMG_DIR)
N_CONCEPTS_TEST = len(test_concepts)

print(f'Image folders | Train: {len(train_concepts)} | Test: {N_CONCEPTS_TEST}')

In [ ]:
TRAIN_CACHE = os.path.join(CACHE_DIR, 'train_clip_embeddings.npy')
TEST_CACHE = os.path.join(CACHE_DIR, 'test_clip_embeddings.npy')

def compute_clip_embeddings(concept_images_list, processor, model, device, desc):
    model.eval()
    all_embeds = []
    with torch.no_grad():
        for imgs in tqdm(concept_images_list, desc=desc):
            for path in imgs:
                inp = processor(images=Image.open(path).convert('RGB'), return_tensors='pt').to(device)
                vo = model.vision_model(**inp)
                feat = F.normalize(model.visual_projection(vo.pooler_output), dim=-1)
                all_embeds.append(feat.cpu().numpy()[0])
    return np.vstack(all_embeds)

if os.path.exists(TRAIN_CACHE) and os.path.exists(TEST_CACHE):
    train_clip_embeds = np.load(TRAIN_CACHE)
    test_clip_embeds = np.load(TEST_CACHE)
    print('Loaded cached CLIP ViT-L/14 embeddings.')
else:
    print('Computing CLIP ViT-L/14 embeddings (one-time)...')
    clip_device = DEVICE
    clip_model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to(clip_device)
    clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
    train_clip_embeds = compute_clip_embeddings(train_concept_imgs, clip_processor, clip_model, clip_device, 'Train')
    test_clip_embeds = compute_clip_embeddings(test_concept_imgs, clip_processor, clip_model, clip_device, 'Test')
    np.save(TRAIN_CACHE, train_clip_embeds)
    np.save(TEST_CACHE, test_clip_embeds)
    print('Saved to cache.')
    del clip_model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

print(f'CLIP ViT-L/14 | Train: {train_clip_embeds.shape} | Test: {test_clip_embeds.shape}')

### Zero-Leakage Dataset & Concept-Level Validation SplitWe carve out **154 concepts (1,540 samples, ~10%)** from the training pool as a strict validation set.- **Concept Prototype Averaging:** The 10 validation trials/images per concept are averaged into 154 concept-level prototypes `(154, 100, 17)` and `(154, 768)`. This aligns validation Top-K (154-way concept retrieval) with Test Top-K (200-way concept retrieval) and accelerates per-epoch evaluation by 100x.- **Test set (200 concepts):** Strictly untouched until the final benchmark table.

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, eeg_data, clip_embeds, augment=False, samples_per_concept=10):
        self.eeg = torch.tensor(eeg_data, dtype=torch.float32)
        self.embeds = torch.tensor(clip_embeds, dtype=torch.float32)
        self.concept_ids = torch.arange(len(eeg_data)) // samples_per_concept
        self.augment = augment

    def __len__(self):
        return len(self.eeg)

    def __getitem__(self, idx):
        x = self.eeg[idx].clone()  # (100, 17)

        if self.augment:
            # 1. Gaussian noise injection (sigma=0.05)
            x = x + torch.randn_like(x) * NOISE_SIGMA

            # 2. Temporal masking: zero out a contiguous block of timepoints
            mask_len = int(x.shape[0] * MASK_RATIO)  # 15 timepoints
            if mask_len > 0:
                start = torch.randint(0, x.shape[0] - mask_len, (1,)).item()
                x[start:start + mask_len, :] = 0.0

        return x, self.embeds[idx], self.concept_ids[idx]

val_eeg_raw = train_eeg[-VAL_SAMPLES:]
val_clip_raw = train_clip_embeds[-VAL_SAMPLES:]

# Average the 10 trials/images per validation concept into 154 concept-level prototypes
# Aligns validation Top-K (154-way concept retrieval) with Test Top-K (200-way concept retrieval)
val_eeg = val_eeg_raw.reshape(VAL_CONCEPTS, 10, N_TIMEPOINTS, N_CHANNELS).mean(axis=1)  # (154, 100, 17)
val_clip_embeds = val_clip_raw.reshape(VAL_CONCEPTS, 10, CLIP_DIM).mean(axis=1)
val_clip_embeds = val_clip_embeds / (np.linalg.norm(val_clip_embeds, axis=-1, keepdims=True) + 1e-8)

train_eeg_sub = train_eeg[:-VAL_SAMPLES]
train_clip_sub = train_clip_embeds[:-VAL_SAMPLES]

train_dataset = EEGDataset(train_eeg_sub, train_clip_sub, augment=True, samples_per_concept=10)
val_dataset = EEGDataset(val_eeg, val_clip_embeds, augment=False, samples_per_concept=1)
test_dataset = EEGDataset(test_eeg, test_clip_embeds, augment=False, samples_per_concept=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples: {len(train_dataset)} ({len(train_dataset)//10} concepts, augmented)')
print(f'Val concepts:  {len(val_dataset)} (154 concept prototypes, clean)')
print(f'Test concepts: {len(test_dataset)} (200 concepts, hold-out)')

### Dual-Objective Loss with False-Negative Masking1. **Cosine Alignment Loss:** Enforces absolute angular tightness against the target.2. **Collision-Masked InfoNCE:** Prevents penalizing false negatives when two images from the same concept appear in the same batch.

In [ ]:
def cosine_loss(pred, target):
    """1 - cosine_similarity. Range: [0, 2]. Lower = better alignment."""
    return 1.0 - F.cosine_similarity(pred, target, dim=-1).mean()

def infonce_loss(pred, target, concept_ids=None, temperature=TEMPERATURE):
    """Symmetric contrastive loss with in-batch false-negative masking."""
    logits = torch.mm(pred, target.T) / temperature
    if concept_ids is not None:
        same_concept = (concept_ids.unsqueeze(1) == concept_ids.unsqueeze(0))
        mask = same_concept.clone()
        mask.fill_diagonal_(False)
        logits = logits.masked_fill(mask, -1e9)
    labels = torch.arange(len(pred), device=pred.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2.0

def combined_loss(pred, target, concept_ids=None):
    return cosine_loss(pred, target) + infonce_loss(pred, target, concept_ids=concept_ids)

### Training Loop with Validation CheckpointingInstead of saving on minimum training loss (which saves overfitted weights), we save checkpoints whenever **Validation Top-10 Retrieval Accuracy** hits a new high.

In [ ]:
def train_model(model, train_loader, val_eeg, val_clip_embeds, epochs, lr, device, save_path, verbose=True):
    """Train with AdamW + linear warmup + cosine decay + validation accuracy checkpointing."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    warmup = min(WARMUP_EPOCHS, max(1, epochs // 5))
    def lr_lambda(epoch):
        if epoch < warmup:
            return float(epoch + 1) / float(warmup)
        progress = (epoch - warmup) / max(1, epochs - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = []
    best_val_acc = -1.0

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0

        for eeg_b, clip_b, cid_b in train_loader:
            eeg_b, clip_b, cid_b = eeg_b.to(device), clip_b.to(device), cid_b.to(device)
            optimizer.zero_grad()
            pred, _ = model(eeg_b)
            loss = combined_loss(pred, clip_b, concept_ids=cid_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            ep_loss += loss.item()

        scheduler.step()
        avg = ep_loss / len(train_loader)
        history.append(avg)

        # Evaluate on validation set to checkpoint on true generalization
        val_acc, _, _ = evaluate(model, val_eeg, val_clip_embeds, device, ks=(1, 5, 10))
        tag = ''
        if val_acc[10] > best_val_acc:
            best_val_acc = val_acc[10]
            if save_path:
                torch.save(model.state_dict(), save_path)
            tag = ' *'

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f'  Epoch {epoch:3d}/{epochs} | Loss: {avg:.4f} | '
                  f'Val Top-10: {val_acc[10]:.1f}% | LR: {scheduler.get_last_lr()[0]:.6f}{tag}')

    if verbose:
        print(f'  Best Val Top-10: {best_val_acc:.1f}%')
    return history

In [ ]:
def evaluate(model, eval_eeg, eval_clip_embeds, device, ks=(1, 5, 10)):
    """Top-K retrieval accuracy via cosine similarity."""
    model.eval()
    with torch.no_grad():
        pred_embeds, attn = model(torch.tensor(eval_eeg, dtype=torch.float32).to(device))
    pred_embeds = pred_embeds.cpu().numpy()
    attn = attn.cpu().numpy()

    sim = cosine_similarity(pred_embeds, eval_clip_embeds)
    n = len(pred_embeds)
    results = {}
    for k in ks:
        correct = sum(1 for i in range(n) if i in np.argsort(sim[i])[::-1][:k])
        results[k] = correct / n * 100.0
    return results, pred_embeds, attn

### Shared Components**BlockEmbedding:** Splits the 100-timepoint EEG into fixed-size temporal blocks and projects each to the model dimension.**TemporalAttention:** Learns which blocks in the sequence are most informative for visual decoding.

In [ ]:
class BlockEmbedding(nn.Module):
    def __init__(self, n_channels, block_size, d_model):
        super().__init__()
        self.block_size = block_size
        self.proj = nn.Sequential(
            nn.Linear(n_channels * block_size, d_model),
            nn.LayerNorm(d_model),
            nn.GELU()
        )

    def forward(self, x):
        B, T, C = x.shape
        n_blocks = T // self.block_size
        x = x[:, :n_blocks * self.block_size, :]
        x = x.reshape(B, n_blocks, C * self.block_size)
        return self.proj(x)

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

    def forward(self, seq):
        w = F.softmax(self.attn(seq).squeeze(-1), dim=-1)
        ctx = (w.unsqueeze(-1) * seq).sum(dim=1)
        return ctx, w

### Model Architectures (5 variants)We define Unidirectional (LSTM, GRU), Bidirectional (BiLSTM, BiGRU), and Global (Transformer) models.

In [ ]:
# 1. Unidirectional LSTM
class EEGtoClipLSTM(nn.Module):
    def __init__(self, n_channels, block_size, hidden_dim=256, num_layers=3, clip_dim=768, dropout=0.3):
        super().__init__()
        self.embed = BlockEmbedding(n_channels, block_size, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.attn = TemporalAttention(hidden_dim)
        self.proj = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, clip_dim))

    def forward(self, x):
        x = self.embed(x)
        x, _ = self.lstm(x)
        ctx, w = self.attn(x)
        return F.normalize(self.proj(ctx), dim=-1), w

# 2. Bidirectional LSTM
class EEGtoClipBiLSTM(nn.Module):
    def __init__(self, n_channels, block_size, hidden_dim=256, num_layers=3, clip_dim=768, dropout=0.3):
        super().__init__()
        self.embed = BlockEmbedding(n_channels, block_size, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.attn = TemporalAttention(hidden_dim * 2)
        self.proj = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, clip_dim))

    def forward(self, x):
        x = self.embed(x)
        x, _ = self.lstm(x)
        ctx, w = self.attn(x)
        return F.normalize(self.proj(ctx), dim=-1), w

# 3. Unidirectional GRU
class EEGtoClipGRU(nn.Module):
    def __init__(self, n_channels, block_size, hidden_dim=256, num_layers=3, clip_dim=768, dropout=0.3):
        super().__init__()
        self.embed = BlockEmbedding(n_channels, block_size, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.attn = TemporalAttention(hidden_dim)
        self.proj = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, clip_dim))

    def forward(self, x):
        x = self.embed(x)
        x, _ = self.gru(x)
        ctx, w = self.attn(x)
        return F.normalize(self.proj(ctx), dim=-1), w

# 4. Bidirectional GRU
class EEGtoClipBiGRU(nn.Module):
    def __init__(self, n_channels, block_size, hidden_dim=256, num_layers=3, clip_dim=768, dropout=0.3):
        super().__init__()
        self.embed = BlockEmbedding(n_channels, block_size, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.attn = TemporalAttention(hidden_dim * 2)
        self.proj = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, clip_dim))

    def forward(self, x):
        x = self.embed(x)
        x, _ = self.gru(x)
        ctx, w = self.attn(x)
        return F.normalize(self.proj(ctx), dim=-1), w

# 5. Transformer
class EEGtoClipTransformer(nn.Module):
    def __init__(self, n_channels, block_size, d_model=256, n_heads=8, n_layers=4, clip_dim=768, dropout=0.3):
        super().__init__()
        n_blocks = N_TIMEPOINTS // block_size
        self.embed = BlockEmbedding(n_channels, block_size, d_model)
        self.pos = nn.Parameter(torch.randn(1, n_blocks, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.attn = TemporalAttention(d_model)
        self.proj = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, clip_dim))

    def forward(self, x):
        x = self.embed(x) + self.pos
        x = self.encoder(x)
        ctx, w = self.attn(x)
        return F.normalize(self.proj(ctx), dim=-1), w

## Step 1: Architecture-Aware Grid Search (Validation Set Only)RNNs prefer larger blocks (fewer sequential steps to prevent vanishing gradients), whereas Transformers excel with smaller blocks (more tokens for multi-head self-attention).We search separately across biologically grounded windows `[5, 10, 20, 25]` ($50	ext{ms}$ to $250	ext{ms}$) using the **validation set**.

In [ ]:
print('=' * 60)
print('  GRID SEARCH: Finding optimal EEG block sizes on Validation Set')
print('=' * 60)

# 1. RNN Family (using lightweight 2-layer GRU)
print('\n--- 1. RNN Family Grid Search (GRU) ---')
rnn_search_results = {}
for bs in BLOCK_SIZES:
    n_blocks = N_TIMEPOINTS // bs
    print(f'  Block size = {bs} ({bs*10}ms)  →  {n_blocks} steps')
    model = EEGtoClipGRU(n_channels=N_CHANNELS, block_size=bs, hidden_dim=256, num_layers=2, clip_dim=CLIP_DIM, dropout=DROPOUT).to(DEVICE)
    train_model(model, train_loader, val_eeg, val_clip_embeds, SEARCH_EPOCHS, LR, DEVICE, save_path=None, verbose=False)
    acc, _, _ = evaluate(model, val_eeg, val_clip_embeds, DEVICE)
    rnn_search_results[bs] = acc[10]
    print(f'  Val Top-10: {acc[10]:.1f}%')
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

BEST_BS_RNN = max(rnn_search_results, key=rnn_search_results.get)
print(f'  >> Best RNN Block Size: {BEST_BS_RNN} ({BEST_BS_RNN*10}ms)')

# 2. Transformer Family (using lightweight 2-layer Transformer)
print('\n--- 2. Transformer Family Grid Search ---')
trans_search_results = {}
for bs in BLOCK_SIZES:
    n_blocks = N_TIMEPOINTS // bs
    print(f'  Block size = {bs} ({bs*10}ms)  →  {n_blocks} tokens')
    model = EEGtoClipTransformer(n_channels=N_CHANNELS, block_size=bs, d_model=256, n_heads=8, n_layers=2, clip_dim=CLIP_DIM, dropout=DROPOUT).to(DEVICE)
    train_model(model, train_loader, val_eeg, val_clip_embeds, SEARCH_EPOCHS, LR, DEVICE, save_path=None, verbose=False)
    acc, _, _ = evaluate(model, val_eeg, val_clip_embeds, DEVICE)
    trans_search_results[bs] = acc[10]
    print(f'  Val Top-10: {acc[10]:.1f}%')
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

BEST_BS_TRANS = max(trans_search_results, key=trans_search_results.get)
print(f'  >> Best Transformer Block Size: {BEST_BS_TRANS} ({BEST_BS_TRANS*10}ms)')
print(f'\n{"=" * 60}\n  FINAL: RNN BS = {BEST_BS_RNN} | Transformer BS = {BEST_BS_TRANS}\n{"=" * 60}')

## Step 2: Train All 5 Models (Validation Checkpointed)Each model trains for 100 epochs with AdamW, linear warmup, and cosine decay.Weights are checkpointed whenever validation Top-10 retrieval accuracy hits a new maximum.

In [ ]:
models_config = {
    'LSTM':        (EEGtoClipLSTM,        BEST_BS_RNN,   {'hidden_dim': 256, 'num_layers': 3}),
    'BiLSTM':      (EEGtoClipBiLSTM,      BEST_BS_RNN,   {'hidden_dim': 256, 'num_layers': 3}),
    'GRU':         (EEGtoClipGRU,         BEST_BS_RNN,   {'hidden_dim': 256, 'num_layers': 3}),
    'BiGRU':       (EEGtoClipBiGRU,       BEST_BS_RNN,   {'hidden_dim': 256, 'num_layers': 3}),
    'Transformer': (EEGtoClipTransformer, BEST_BS_TRANS, {'d_model': 256, 'n_heads': 8, 'n_layers': 4})
}

histories = {}
accuracies = {}
predictions = {}
param_counts = {}

for name, (ModelClass, bs, extra_kwargs) in models_config.items():
    print(f'=== {name} (block_size={bs}) ===')
    model = ModelClass(
        n_channels=N_CHANNELS, block_size=bs,
        **extra_kwargs,
        clip_dim=CLIP_DIM, dropout=DROPOUT
    ).to(DEVICE)
    
    param_counts[name] = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Parameters: {param_counts[name]:,}')
    
    ckpt_path = os.path.join(CKPT_DIR, f'{name.lower()}_best.pt')
    histories[name] = train_model(model, train_loader, val_eeg, val_clip_embeds, EPOCHS, LR, DEVICE, ckpt_path)
    
    # Load best checkpoint based on validation performance and evaluate strictly on test set
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    acc, preds, attn = evaluate(model, test_eeg, test_clip_embeds, DEVICE)
    
    accuracies[name] = acc
    predictions[name] = preds
    print(f'  [TEST] Top-1: {acc[1]:.1f}%  |  Top-5: {acc[5]:.1f}%  |  Top-10: {acc[10]:.1f}%\n')
    
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## Step 3: Final Test Set Benchmark ComparisonStrict zero-shot retrieval across the 200 hold-out test concepts.

In [ ]:
random_1 = 100 / N_CONCEPTS_TEST
random_5 = 500 / N_CONCEPTS_TEST
random_10 = 1000 / N_CONCEPTS_TEST

header = f"{'Metric':<14} | {'LSTM':>8} | {'BiLSTM':>8} | {'GRU':>8} | {'BiGRU':>8} | {'Trans.':>8} | {'Random':>8}"
print(header)
print("-" * len(header))

print(f"{'Parameters':<14} | {param_counts['LSTM']:>8,} | {param_counts['BiLSTM']:>8,} | {param_counts['GRU']:>8,} | {param_counts['BiGRU']:>8,} | {param_counts['Transformer']:>8,} | {'—':>8}")

for k, rand in zip([1, 5, 10], [random_1, random_5, random_10]):
    row = f"Top-{k:<10} | "
    for name in ['LSTM', 'BiLSTM', 'GRU', 'BiGRU', 'Transformer']:
        row += f"{accuracies[name][k]:>7.1f}% | "
    row += f"{rand:>7.1f}%"
    print(row)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = {'LSTM': '#3B82F6', 'BiLSTM': '#1E3A8A', 'GRU': '#10B981', 'BiGRU': '#064E3B', 'Transformer': '#EF4444'}

for name, hist in histories.items():
    ax.plot(hist, color=colors[name], linewidth=2, label=name)

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (Cosine + InfoNCE)')
ax.set_title('Training Loss Curves (5 Models)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'training_loss_5models.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ks = [1, 5, 10]
x = np.arange(len(ks))
w = 0.14
offsets = [-2.5, -1.5, -0.5, 0.5, 1.5, 2.5]
models = ['LSTM', 'BiLSTM', 'GRU', 'BiGRU', 'Transformer']
bar_colors = [colors[m] for m in models] + ['#9CA3AF']

fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(models):
    ax.bar(x + offsets[i]*w, [accuracies[name][k] for k in ks], w, label=name, color=bar_colors[i], alpha=0.85)

ax.bar(x + offsets[-1]*w, [random_1, random_5, random_10], w, label='Random', color=bar_colors[-1], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(['Top-1', 'Top-5', 'Top-10'])
ax.set_ylabel('Accuracy (%)')
ax.set_title('Top-K Test Retrieval Accuracy Comparison (200 Concepts)', fontweight='bold')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'accuracy_comparison_5models.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def retrieve_nearest(pred_embed, all_embeds, all_imgs):
    sims = cosine_similarity(pred_embed[np.newaxis], all_embeds)[0]
    idx = np.argmax(sims)
    return idx, sims[idx], all_imgs[idx][0]

DEMO = [0, 5, 10, 20, 30]
n = len(DEMO)

fig, axes = plt.subplots(n, 6, figsize=(18, n * 3.2))
fig.suptitle('Retrieval: GT | LSTM | BiLSTM | GRU | BiGRU | Transformer', fontsize=14, fontweight='bold')

for row, idx in enumerate(DEMO):
    concept = test_concepts[idx]

    # Ground Truth
    gt = Image.open(test_concept_imgs[idx][0]).convert('RGB').resize((224, 224))
    axes[row, 0].imshow(gt)
    axes[row, 0].set_title(f'GT: "{concept}"', fontsize=9, fontweight='bold')
    axes[row, 0].axis('off')

    # Models
    for col, name in enumerate(['LSTM', 'BiLSTM', 'GRU', 'BiGRU', 'Transformer'], 1):
        preds = predictions[name]
        ri, rs, rp = retrieve_nearest(preds[idx], test_clip_embeds, test_concept_imgs)
        axes[row, col].imshow(Image.open(rp).convert('RGB').resize((224, 224)))
        ok = ri == idx
        mark = '\u2713' if ok else '\u2717'
        axes[row, col].set_title(f'{mark} {name}\n"{test_concepts[ri]}"\nSim:{rs:.3f}',
            fontsize=9, fontweight='bold', color='#16A34A' if ok else '#DC2626')
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'retrieval_comparison_5models.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Save Predicted Vectors & Downstream Image Synthesis> **Important (Stable Diffusion Conditioning Architecture):**> Standard Stable Diffusion v1.4/v1.5 text cross-attention expects unprojected sequence tokens of shape `(Batch, 77, 768)`.> Our EEG encoders output a single $L_2$-normalized global projected image embedding of shape `(Batch, 768)` from `model.visual_projection`.>> To synthesize images from these exported `.npy` vectors, use an image-conditioned diffusion pipeline designed for pooled CLIP embeddings:> 1. **`StableDiffusionImageVariationPipeline`** (`lambdalabs/sd-image-variations-diffusers`): directly accepts `image_embeddings` `[1, 768]`.> 2. **`StableDiffusionPipeline` + `IP-Adapter`**: uses cross-attention resamplers to map the pooled 768-d vector into the UNet.> 3. **`Versatile Diffusion`** or **`unCLIP` (`stabilityai/stable-diffusion-2-1-unclip`)**.

In [ ]:
best_name = max(accuracies, key=lambda k: accuracies[k][10])

for name in ['LSTM', 'BiLSTM', 'GRU', 'BiGRU', 'Transformer']:
    np.save(os.path.join(VEC_DIR, f'{name.lower()}_pred_embeddings.npy'), predictions[name])
    
np.save(os.path.join(VEC_DIR, 'test_clip_gt_embeddings.npy'), test_clip_embeds)
np.save(os.path.join(VEC_DIR, 'test_concept_names.npy'), np.array(test_concepts))

print(f'Vectors saved to {VEC_DIR}/')
print(f'Vector dimension: {CLIP_DIM} (native CLIP ViT-L/14 image space)')
print(f'Best Model on Test Set (Top-10): {best_name} ({accuracies[best_name][10]:.1f}%)')